In [22]:
import base64
from mistralai import Mistral

client = Mistral(api_key="fbRcIZTHmtHrWaTNnqUVfTUPWHzhebh3")

image_path = "/Users/asseromar/Library/CloudStorage/OneDrive-aivancity/PGE 4/Alternance/Data Extraction/mistral-ocr/test12.jpg"

with open(image_path, "rb") as f:
    image_b64 = base64.b64encode(f.read()).decode("utf-8")

prompt = """
Tu es un assistant d'analyse de documents.

Analyse attentivement l'image du document (facture, devis, reçu, bon de commande, etc.)
et retourne UNIQUEMENT le JSON suivant avec les valeurs trouvées.

Même si les intitulés diffèrent, identifie les champs équivalents :

- "numero_de_dossier" → peut être appelé "Référence", "N° Dossier", "Réf", "N/REF", etc. , Ce numéro peut parfois contenir un point (par ex. "2025.01" ou "1234546.01-PLAN")
- "numero_de_facture" → peut être appelé "Facture N°", "Invoice No", "N", etc.
- "date_de_facture" → peut être appelé "Date", "Invoice Date", "Date d’émission", etc.
  ⚠️ Si le document ne contient qu’une période (ex: "du 01/07/2023 au 30/08/2023"),
  ne considère pas ces dates comme la date de facture.
  Dans ce cas, laisse "date_de_facture" vide ("").
- "montant_ht" → peut être appelé "Montant HT", "Net Amount", "Subtotal", "Total (excl. tax)", etc.
- "montant_tva" → peut être appelé "TVA", "VAT", "Tax", "Tax Amount", etc.
- "montant_ttc" → peut être appelé "Montant TTC", "Total TTC", "Amount Due", "Total (incl. tax)", etc.

Si une valeur est absente, laisse-la vide ("").

Retourne UNIQUEMENT un JSON propre et valide, dans le format suivant :

{
  "numero_de_dossier": "",
  "numero_de_facture": "",
  "date_de_facture": "",
  "montant_ht": "",
  "montant_tva": "",
  "montant_ttc": ""
}
"""

response = client.chat.complete(
    model="pixtral-12b",
    messages=[
        {
            "role": "user",
            "content": [
                {"type": "image_url", "image_url": f"data:image/jpeg;base64,{image_b64}"},
                {"type": "text", "text": prompt}
            ]
        }
    ],
    temperature=0.2,
    max_tokens=400
)

print("OCR Extraction Result:")
print(response.choices[0].message.content)


OCR Extraction Result:
```json
{
  "numero_de_dossier": "3925008611.01 - APFSE",
  "numero_de_facture": "25.000977",
  "date_de_facture": "28/04/2025",
  "montant_ht": "675.08 €",
  "montant_tva": "135.02 €",
  "montant_ttc": "810.10 €"
}
```
